# Layer 1 — Exploratory Data Analysis
## MerRec C2C Interaction Dataset

**Capstone Project** | American University of Armenia  
**Student:** Arevik Melikyan  
**Supervisor:** Varazdat Stepanyan

---

### Overview

This notebook constitutes **Layer 1** of the analytical framework: a comprehensive exploratory data analysis of the MerRec dataset (~1 billion implicit C2C interactions). The analysis is structured across ten sections that together characterise the dataset's structural properties, behavioral distributions, temporal dynamics, and data quality — forming the evidence base for all modelling decisions in subsequent layers.

**Dataset columns:** `user_id`, `item_id`, `session_id`, `stime`, `event_id`, `price`, `c0_name`, `c1_name`, `c2_name`

**Sections**
1. Environment setup & schema audit  
2. Dataset-level summary statistics  
3. User-side distributions  
4. Item-side distributions  
5. Session-level statistics  
6. Event & funnel analysis  
7. Category hierarchy analysis  
8. Temporal patterns  
9. Price distribution  
10. Sparsity, Gini & cold-start diagnostics  


---
## 1. Environment Setup & Schema Audit

All heavy aggregations run inside **DuckDB** (in-memory, 10 threads, 20 GB cap) via a lazy view over the Parquet files. This avoids loading the full dataset into RAM while enabling SQL-style analytics at scale.

Figures are saved to `./eda_outputs/` so every result is reproducible and portable.


In [1]:
import os
import warnings
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Paths ─────────────────────────────────────────────────────────────────────
DATASET_ROOT = Path(os.environ.get("DATASET_ROOT", "/Volumes/T5 EVO"))
DATASET_PATH = DATASET_ROOT / "hf" / "merrec"
OUTPUT_DIR   = Path("eda_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── DuckDB ────────────────────────────────────────────────────────────────────
con = duckdb.connect(database=":memory:")
con.execute("PRAGMA threads=10")
con.execute("PRAGMA memory_limit='20GB'")

con.execute(f"""
CREATE VIEW merrec AS
SELECT *
FROM read_parquet('{DATASET_PATH}/**/*.parquet')
""")

print("View created successfully.")
print("Dataset path:", DATASET_PATH)


View created successfully.
Dataset path: /Volumes/T5 EVO/hf/merrec


In [2]:
# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", font_scale=1.05)
PALETTE = sns.color_palette("tab10")

def save(name: str):
    """Apply tight layout, save to OUTPUT_DIR, and display inline."""
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  ✓ saved → eda_outputs/{name}.png")


In [3]:
# ── Schema inspection ────────────────────────────────────────────────────────
schema = con.execute("DESCRIBE merrec").df()
print("Dataset schema:")
display(schema)


Dataset schema:


,column_name,column_type,null,key,default,extra
0,user_id,BIGINT,YES,None,None,None
1,stime,TIMESTAMP,YES,None,None,None
2,session_id,VARCHAR,YES,None,None,None
3,sequence_id,VARCHAR,YES,None,None,None
4,sequence_length,BIGINT,YES,None,None,None
5,event_id,VARCHAR,YES,None,None,None
6,item_id,BIGINT,YES,None,None,None
7,product_id,VARCHAR,YES,None,None,None
8,name,VARCHAR,YES,None,None,None
9,price,DOUBLE,YES,None,None,None


---
## 2. Dataset-Level Summary Statistics

Before drilling into individual dimensions, we establish a single-query global summary that captures:
- Scale (events, users, items, sessions)
- Temporal extent (earliest / latest event, total span in days)
- Missing-value counts per column — critical for downstream imputation decisions


In [ ]:
summary = con.execute("""
SELECT
    COUNT(*)                                   AS total_events,
    APPROX_COUNT_DISTINCT(user_id)             AS approx_users,
    APPROX_COUNT_DISTINCT(item_id)             AS approx_items,
    APPROX_COUNT_DISTINCT(session_id)          AS approx_sessions,
    COUNT(DISTINCT event_id)                   AS event_types,
    MIN(stime)                                 AS earliest,
    MAX(stime)                                 AS latest,
    DATEDIFF('day', MIN(stime), MAX(stime))    AS span_days,
    COUNT(*) FILTER (WHERE price    IS NULL)   AS null_price,
    COUNT(*) FILTER (WHERE c0_name  IS NULL)   AS null_c0,
    COUNT(*) FILTER (WHERE c1_name  IS NULL)   AS null_c1,
    COUNT(*) FILTER (WHERE c2_name  IS NULL)   AS null_c2,
    COUNT(*) FILTER (WHERE user_id  IS NULL)   AS null_user,
    COUNT(*) FILTER (WHERE item_id  IS NULL)   AS null_item
FROM merrec
""").df()

print("=== Global Dataset Summary ===")
display(summary.T.rename(columns={0: "value"}))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
# ── Missing-value audit ───────────────────────────────────────────────────────
null_cols = ["null_price", "null_c0", "null_c1", "null_c2", "null_user", "null_item"]
null_vals = summary[null_cols].values.flatten()
total     = summary["total_events"].values[0]
null_pct  = null_vals / total * 100

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(null_cols, null_pct, color=PALETTE[:len(null_cols)])
ax.set_xlabel("Missing (%)")
ax.set_title("Missing-Value Audit by Column")
for bar, pct in zip(bars, null_pct):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
            f"{pct:.1f}%", va="center", fontsize=9)
save("01_missing_value_audit")


---
## 3. User-Side Distributions

This section characterises the user population across five dimensions:

| Plot | Purpose |
|------|---------|
| Events per user (log-log histogram + CCDF) | Quantify long-tail structure and power-law behaviour |
| User lifespan | Distinguish casual one-time visitors from long-term users |
| Engagement vs lifespan scatter | Identify volume vs longevity trade-offs |
| Category entropy | Measure browsing focus (specialist vs explorer) |
| New vs returning users (weekly) | Track user retention and platform growth over time |


In [ ]:
# ── 3a. Events per user (full scan, exact counts) ─────────────────────────────
events_per_user = con.execute("""
    SELECT user_id, COUNT(*) AS event_count
    FROM merrec
    GROUP BY user_id
""").fetch_arrow_table()

ec = np.array(events_per_user["event_count"])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(ec, bins=np.logspace(0, np.log10(ec.max()), 60),
             color=PALETTE[0], edgecolor="none", alpha=0.85)
axes[0].set(xscale="log", yscale="log",
            xlabel="Events per user (log)",
            ylabel="Number of users (log)",
            title="Events per User — Log-Log Histogram")

sorted_ec = np.sort(ec)
ccdf = 1.0 - np.arange(1, len(sorted_ec) + 1) / len(sorted_ec)
axes[1].plot(sorted_ec, ccdf, linewidth=1.3, color=PALETTE[0])
axes[1].set(xscale="log", yscale="log",
            xlabel="Events per user (log)",
            ylabel="P(X > x)",
            title="CCDF of Events per User")

fig.suptitle("User Activity Distribution", fontsize=13, fontweight="bold", y=1.02)
save("02a_events_per_user")


In [ ]:
# ── 3b. User lifespan ─────────────────────────────────────────────────────────
lifespan = con.execute("""
    SELECT EXTRACT(EPOCH FROM (MAX(stime) - MIN(stime))) / 86400.0 AS days
    FROM merrec
    GROUP BY user_id
""").df()

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(lifespan["days"].dropna(), bins=60, color=PALETTE[1], edgecolor="none", alpha=0.85)
ax.set(xlabel="Active lifespan (days)",
       ylabel="Number of users",
       title="User Activity Lifespan Distribution")
med = lifespan["days"].median()
ax.axvline(med, color="crimson", linestyle="--", linewidth=1.2)
ax.text(med + 1, ax.get_ylim()[1] * 0.9, f"median = {med:.0f} days",
        color="crimson", fontsize=9)
save("02b_user_lifespan")


In [ ]:
# ── 3c. Interactions vs lifespan ──────────────────────────────────────────────
engagement = con.execute("""
    SELECT
        COUNT(*) AS interactions,
        EXTRACT(EPOCH FROM (MAX(stime) - MIN(stime))) / 86400.0 AS span
    FROM merrec
    GROUP BY user_id
""").df()

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(engagement["span"], engagement["interactions"],
           s=3, alpha=0.25, color=PALETTE[2], rasterized=True)
ax.set(yscale="log",
       xlabel="Active span (days)",
       ylabel="Total interactions (log)",
       title="User Engagement vs Lifespan")
save("02c_user_engagement_vs_lifespan")


In [ ]:
# ── 3d. Category entropy per user ────────────────────────────────────────────
#
# Shannon entropy of a user's category distribution (Level 0).
# H = 0  → all interactions in a single category (specialist)
# H = max → perfectly uniform across all categories (explorer)

entropy_df = con.execute("""
    WITH uc AS (
        SELECT user_id, c0_name, COUNT(*) AS cnt
        FROM merrec
        WHERE c0_name IS NOT NULL
        GROUP BY user_id, c0_name
    ),
    totals AS (
        SELECT user_id, SUM(cnt) AS total FROM uc GROUP BY user_id
    )
    SELECT
        uc.user_id,
        -SUM((uc.cnt * 1.0 / t.total) * LOG2(uc.cnt * 1.0 / t.total)) AS entropy
    FROM uc JOIN totals t USING(user_id)
    GROUP BY uc.user_id
""").df()

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(entropy_df["entropy"].dropna(), bins=60, color=PALETTE[3], edgecolor="none", alpha=0.85)
ax.set(xlabel="Category entropy (bits)",
       ylabel="Number of users",
       title="User Preference Entropy (Category Level 0)")
med_e = entropy_df["entropy"].median()
ax.axvline(med_e, color="crimson", linestyle="--", linewidth=1.2)
ax.text(med_e + 0.05, ax.get_ylim()[1] * 0.9,
        f"median = {med_e:.2f} bits", color="crimson", fontsize=9)
save("02d_user_category_entropy")


In [ ]:
# ── 3e. New vs returning users (weekly) ──────────────────────────────────────
new_ret = con.execute("""
    WITH first_seen AS (
        SELECT user_id, MIN(DATE_TRUNC('week', stime)) AS first_week
        FROM merrec GROUP BY user_id
    ),
    weekly_users AS (
        SELECT DATE_TRUNC('week', stime) AS week, user_id
        FROM merrec GROUP BY week, user_id
    )
    SELECT
        wu.week,
        COUNT(*) FILTER (WHERE wu.week = fs.first_week) AS new_users,
        COUNT(*) FILTER (WHERE wu.week > fs.first_week)  AS returning_users
    FROM weekly_users wu
    JOIN first_seen fs USING(user_id)
    GROUP BY wu.week ORDER BY wu.week
""").df()

new_ret["week"] = pd.to_datetime(new_ret["week"])

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(new_ret["week"], new_ret["new_users"],       label="New users",      linewidth=1.3)
ax.plot(new_ret["week"], new_ret["returning_users"], label="Returning users", linewidth=1.3)
ax.set(yscale="log", xlabel="Week", ylabel="Users (log)",
       title="New vs Returning Users per Week")
ax.legend()
save("02e_new_vs_returning_users")


---
## 4. Item-Side Distributions

The item distribution analysis mirrors the user analysis and is equally important: in C2C marketplaces the item catalog is dynamic (sellers continuously add and remove listings), leading to strong temporal and popularity heterogeneity.

| Plot | Purpose |
|------|---------|
| Events per item (log-log + CCDF) | Quantify item-side long-tail; informs item cold-start severity |
| Item lifespan vs total events | Distinguish viral short-lived items from long-tail persistent ones |


In [ ]:
# ── 4a. Events per item ───────────────────────────────────────────────────────
events_per_item = con.execute("""
    SELECT item_id, COUNT(*) AS event_count
    FROM merrec
    GROUP BY item_id
""").fetch_arrow_table()

ic = np.array(events_per_item["event_count"])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(ic, bins=np.logspace(0, np.log10(ic.max()), 60),
             color=PALETTE[4], edgecolor="none", alpha=0.85)
axes[0].set(xscale="log", yscale="log",
            xlabel="Events per item (log)",
            ylabel="Number of items (log)",
            title="Events per Item — Log-Log Histogram")

sorted_ic = np.sort(ic)
ccdf_i = 1.0 - np.arange(1, len(sorted_ic) + 1) / len(sorted_ic)
axes[1].plot(sorted_ic, ccdf_i, linewidth=1.3, color=PALETTE[4])
axes[1].set(xscale="log", yscale="log",
            xlabel="Events per item (log)",
            ylabel="P(X > x)",
            title="CCDF of Events per Item")

fig.suptitle("Item Activity Distribution", fontsize=13, fontweight="bold", y=1.02)
save("03a_events_per_item")


In [ ]:
# ── 4b. Item lifespan vs total events ────────────────────────────────────────
item_lifespan = con.execute("""
    SELECT
        item_id,
        COUNT(*) AS total_events,
        DATEDIFF('day', MIN(DATE(stime)), MAX(DATE(stime))) AS span_days
    FROM merrec
    GROUP BY item_id
""").df()

item_lifespan["span_days"] = item_lifespan["span_days"].clip(lower=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(
    item_lifespan["span_days"],
    bins=np.logspace(0, np.log10(item_lifespan["span_days"].max()), 50),
    color=PALETTE[5], edgecolor="none", alpha=0.85
)
axes[0].set(xscale="log", yscale="log",
            xlabel="Item lifespan (days, log)",
            ylabel="Number of items (log)",
            title="Item Attention Lifespan")

axes[1].scatter(item_lifespan["span_days"], item_lifespan["total_events"],
                s=3, alpha=0.2, color=PALETTE[5], rasterized=True)
axes[1].set(xscale="log", yscale="log",
            xlabel="Lifespan (days, log)",
            ylabel="Total events (log)",
            title="Item Lifespan vs Total Events")

fig.suptitle("Item Lifespan Analysis", fontsize=13, fontweight="bold", y=1.02)
save("03b_item_lifespan")


---
## 5. Session-Level Statistics

Sessions are the primary unit of short-term intent. Understanding session structure directly informs the research question: *how do short-term session dynamics compare to long-term user preferences in determining item relevance?*

| Metric | Interpretation |
|--------|---------------|
| Session length | Depth of engagement per visit |
| Session duration | Time investment per visit |
| Unique items per session | Browsing breadth within a visit |
| Sessions per user | Visit frequency proxy |
| Bounce rate | Share of single-event sessions (immediate exits) |


In [ ]:
# Pre-compute session-level aggregates once and reuse across all plots
session_stats = con.execute("""
    SELECT
        session_id,
        user_id,
        COUNT(*)                                                    AS session_len,
        DATEDIFF('second', MIN(stime), MAX(stime))                 AS duration_sec,
        COUNT(DISTINCT item_id)                                    AS unique_items,
        COUNT(DISTINCT event_id)                                   AS event_types_used,
        COUNT(*) FILTER (WHERE event_id = 'item_view')            AS views,
        COUNT(*) FILTER (WHERE event_id = 'buy_comp')             AS purchases
    FROM merrec
    GROUP BY session_id, user_id
""").df()

print(f"Sessions computed: {len(session_stats):,}")
display(session_stats.describe().round(2))


In [ ]:
# ── 5a. Session length distribution ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(session_stats["session_len"].clip(upper=200), bins=60,
             color=PALETTE[6], edgecolor="none", alpha=0.85)
axes[0].set(xlabel="Events per session (clipped at 200)",
            ylabel="Number of sessions",
            title="Session Length — Linear")

axes[1].hist(session_stats["session_len"],
             bins=np.logspace(0, np.log10(session_stats["session_len"].max()), 50),
             color=PALETTE[6], edgecolor="none", alpha=0.85)
axes[1].set(xscale="log", yscale="log",
            xlabel="Events per session (log)",
            ylabel="Number of sessions (log)",
            title="Session Length — Log-Log")

fig.suptitle("Session Length Distribution", fontsize=13, fontweight="bold", y=1.02)
save("04a_session_length")


In [ ]:
# ── 5b. Session duration ──────────────────────────────────────────────────────
dur_min = session_stats["duration_sec"].dropna() / 60

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(dur_min.clip(upper=120), bins=60, color=PALETTE[7], edgecolor="none", alpha=0.85)
ax.set(xlabel="Session duration (minutes, clipped at 120)",
       ylabel="Number of sessions",
       title="Session Duration Distribution")
med_dur = dur_min.median()
ax.axvline(med_dur, color="crimson", linestyle="--", linewidth=1.2)
ax.text(med_dur + 0.5, ax.get_ylim()[1] * 0.9,
        f"median = {med_dur:.1f} min", color="crimson", fontsize=9)
save("04b_session_duration")


In [ ]:
# ── 5c. Unique items per session ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(session_stats["unique_items"].clip(upper=100), bins=60,
        color=PALETTE[8], edgecolor="none", alpha=0.85)
ax.set(xlabel="Unique items per session (clipped at 100)",
       ylabel="Number of sessions",
       title="Item Breadth per Session")
save("04c_items_per_session")


In [ ]:
# ── 5d. User session behaviour space ─────────────────────────────────────────
#
# Each point = one user. Axes: number of sessions vs average session length.
# This diagnostic view reveals whether users cluster into distinguishable
# behavioral modes (explorers, focused buyers, passive browsers) —
# a preview of the regime analysis in Layer 3.

user_session_agg = con.execute("""
    SELECT
        user_id,
        APPROX_COUNT_DISTINCT(session_id)                               AS num_sessions,
        (COUNT(*) * 1.0) / NULLIF(APPROX_COUNT_DISTINCT(session_id), 0) AS avg_session_len
    FROM merrec
    GROUP BY user_id
""").df()

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(user_session_agg["num_sessions"], user_session_agg["avg_session_len"],
           s=4, alpha=0.25, color=PALETTE[9], rasterized=True)
ax.set(xscale="log", yscale="log",
       xlabel="Number of sessions (log)",
       ylabel="Avg session length (log)",
       title="User Session Behaviour Space\n(each point = one user)")
save("04d_user_session_behaviour")


In [ ]:
# ── 5e. Session bounce rate ───────────────────────────────────────────────────
bounce_rate = (session_stats["session_len"] == 1).mean()
multi_pct   = 1 - bounce_rate

fig, ax = plt.subplots(figsize=(5, 5))
ax.pie([bounce_rate, multi_pct],
       labels=[f"Bounce (1 event)\n{bounce_rate*100:.1f}%",
               f"Multi-event\n{multi_pct*100:.1f}%"],
       colors=[PALETTE[0], PALETTE[2]],
       startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 1.5})
ax.set_title("Session Bounce Rate")
save("04e_session_bounce_rate")


---
## 6. Event & Funnel Analysis

Implicit feedback in C2C platforms is multi-typed. Understanding the frequency of each event type and the conversion rates between them quantifies the observability of user intent and directly informs how interaction signals should be weighted in latent factor models.

The funnel analysis tracks unique `(user, item)` pairs at each conversion stage: **View → Like → Add to Cart → Purchase**.


In [ ]:
# ── 6a. Event type distribution ──────────────────────────────────────────────
event_dist = con.execute("""
    SELECT event_id, COUNT(*) AS cnt
    FROM merrec
    GROUP BY event_id
    ORDER BY cnt DESC
""").df()

fig, ax = plt.subplots(figsize=(9, max(4, len(event_dist) * 0.45)))
ax.barh(event_dist["event_id"][::-1], event_dist["cnt"][::-1],
        color=PALETTE[0], edgecolor="none", alpha=0.85)
ax.set(xscale="log", xlabel="Count (log)", title="Event Type Distribution")
save("05a_event_type_distribution")


In [ ]:
# ── 6b. Funnel drop-off ───────────────────────────────────────────────────────
funnel = con.execute("""
    SELECT
        APPROX_COUNT_DISTINCT(CONCAT_WS(',',
            CAST(user_id AS VARCHAR), CAST(item_id AS VARCHAR)))
            FILTER (WHERE event_id = 'item_view')             AS views,
        APPROX_COUNT_DISTINCT(CONCAT_WS(',',
            CAST(user_id AS VARCHAR), CAST(item_id AS VARCHAR)))
            FILTER (WHERE event_id = 'item_like')             AS likes,
        APPROX_COUNT_DISTINCT(CONCAT_WS(',',
            CAST(user_id AS VARCHAR), CAST(item_id AS VARCHAR)))
            FILTER (WHERE event_id = 'item_add_to_cart_tap')  AS carts,
        APPROX_COUNT_DISTINCT(CONCAT_WS(',',
            CAST(user_id AS VARCHAR), CAST(item_id AS VARCHAR)))
            FILTER (WHERE event_id = 'buy_comp')              AS purchases
    FROM merrec
""").df()

stages      = ["View", "Like", "Add to Cart", "Purchase"]
funnel_vals = [funnel["views"].iloc[0], funnel["likes"].iloc[0],
               funnel["carts"].iloc[0], funnel["purchases"].iloc[0]]

drop_rates = [1.0] + [funnel_vals[i] / funnel_vals[i-1]
                       for i in range(1, len(funnel_vals))]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(stages, funnel_vals, marker="o", linewidth=1.5, color=PALETTE[1])
axes[0].set(yscale="log", ylabel="User–Item Pairs (log)",
            title="Interaction Funnel (log scale)")
axes[0].tick_params(axis="x", rotation=15)

axes[1].bar(stages, [r * 100 for r in drop_rates],
            color=PALETTE[2], edgecolor="none", alpha=0.85)
axes[1].set(ylabel="Retention vs previous stage (%)",
            title="Funnel Retention Rate per Stage")
axes[1].axhline(100, color="gray", linewidth=0.8, linestyle="--")
axes[1].tick_params(axis="x", rotation=15)

fig.suptitle("Interaction Funnel Analysis", fontsize=13, fontweight="bold", y=1.02)
save("05b_funnel_dropoff")


In [ ]:
# ── 6c. Event-type transition matrix ─────────────────────────────────────────
#
# Row-normalised to conditional probabilities P(next event | current event).
# Sampled on 20,000 users from the last 30 days to remain memory-efficient.

event_trans = con.execute("""
    WITH recent_bounds AS (
        SELECT MAX(stime) - INTERVAL '30 days' AS start FROM merrec
    ),
    sampled_users AS (
        SELECT user_id
        FROM merrec, recent_bounds
        WHERE merrec.stime >= recent_bounds.start
        GROUP BY user_id ORDER BY RANDOM() LIMIT 20000
    ),
    ordered AS (
        SELECT user_id, event_id,
               LEAD(event_id) OVER (PARTITION BY user_id ORDER BY stime) AS next_event
        FROM merrec
        WHERE user_id IN (SELECT user_id FROM sampled_users)
    )
    SELECT event_id AS from_event, next_event AS to_event,
           APPROX_COUNT_DISTINCT(user_id) AS cnt
    FROM ordered
    WHERE next_event IS NOT NULL
    GROUP BY from_event, to_event
""").df()

unique_events = sorted(set(event_trans["from_event"]) | set(event_trans["to_event"]))
eidx = {e: i for i, e in enumerate(unique_events)}
mat  = np.zeros((len(unique_events), len(unique_events)))
for _, row in event_trans.iterrows():
    f, t = row["from_event"], row["to_event"]
    if f in eidx and t in eidx:
        mat[eidx[f], eidx[t]] = row["cnt"]

row_sums = mat.sum(axis=1, keepdims=True)
mat_norm = np.divide(mat, row_sums, where=row_sums > 0)

n = len(unique_events)
fig, ax = plt.subplots(figsize=(max(7, n), max(6, n - 1)))
im = ax.imshow(mat_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(unique_events, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(unique_events, fontsize=8)
plt.colorbar(im, ax=ax, label="P(next event | current event)")
ax.set_title("Event-Type Transition Matrix — Row-Normalised\n(sampled 20k users, last 30 days)")
save("05c_event_transition_matrix")


---
## 7. Category Hierarchy Analysis

The dataset has a three-level category hierarchy (`c0_name` → `c1_name` → `c2_name`). Category structure is important for three reasons:

1. **Cold-start mitigation** — category context can provide item representations even with few direct interactions.
2. **Behavioral interpretability** — category preferences are a natural human-readable segment attribute.
3. **Feature engineering** — category-level interaction rates will be core features in Layer 2.

This section covers frequency distributions at all three levels and a category-to-category transition matrix at Level 0.


In [ ]:
# ── 7a. Top categories at each hierarchy level ───────────────────────────────
for level, col, topk in [("L0", "c0_name", 20), ("L1", "c1_name", 30), ("L2", "c2_name", 30)]:
    cat_df = con.execute(f"""
        SELECT {col}, COUNT(*) AS cnt
        FROM merrec
        WHERE {col} IS NOT NULL
        GROUP BY {col}
        ORDER BY cnt DESC
        LIMIT {topk}
    """).df()

    fig, ax = plt.subplots(figsize=(10, max(5, len(cat_df) * 0.35)))
    ax.barh(cat_df[col][::-1], cat_df["cnt"][::-1],
            color=PALETTE[1], edgecolor="none", alpha=0.85)
    ax.set(xscale="log", xlabel="Number of events (log)",
           title=f"Top {topk} Categories — {level}")
    save(f"06_top_categories_{level}")


In [ ]:
# ── 7b. Category-to-category transition matrix (L0, top-10) ──────────────────
top10_cats = con.execute("""
    SELECT c0_name FROM (
        SELECT c0_name, COUNT(*) AS cnt
        FROM merrec WHERE c0_name IS NOT NULL
        GROUP BY c0_name ORDER BY cnt DESC LIMIT 10
    )
""").df()["c0_name"].tolist()

escaped  = [c.replace("'", "''") for c in top10_cats]
cats_sql = "(" + ", ".join(f"'{c}'" for c in escaped) + ")"

cat_trans = con.execute(f"""
    WITH recent_bounds AS (
        SELECT MAX(stime) - INTERVAL '90 days' AS start FROM merrec
    ),
    sampled_users AS (
        SELECT user_id
        FROM merrec, recent_bounds
        WHERE merrec.c0_name IN {cats_sql}
          AND merrec.stime >= recent_bounds.start
        GROUP BY user_id ORDER BY RANDOM() LIMIT 20000
    ),
    ordered AS (
        SELECT user_id, c0_name,
               LEAD(c0_name) OVER (PARTITION BY user_id ORDER BY stime) AS next_cat
        FROM merrec
        WHERE user_id IN (SELECT user_id FROM sampled_users)
          AND c0_name IN {cats_sql}
    )
    SELECT c0_name AS from_cat, next_cat AS to_cat, COUNT(*) AS cnt
    FROM ordered WHERE next_cat IS NOT NULL
    GROUP BY c0_name, next_cat
""").df()

cidx = {{c: i for i, c in enumerate(top10_cats)}}
cmat = np.zeros((len(top10_cats), len(top10_cats)))
for _, row in cat_trans.iterrows():
    f, t = row["from_cat"], row["to_cat"]
    if f in cidx and t in cidx:
        cmat[cidx[f], cidx[t]] = row["cnt"]

crow     = cmat.sum(axis=1, keepdims=True)
cmat_norm = np.divide(cmat, crow, where=crow > 0)

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cmat_norm, cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks(range(len(top10_cats))); ax.set_yticks(range(len(top10_cats)))
ax.set_xticklabels(top10_cats, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(top10_cats, fontsize=8)
plt.colorbar(im, ax=ax, label="P(next category | current category)")
ax.set_title("Category Transition Matrix — Top-10 L0\n(row-normalised, sampled last 90 days)")
save("06b_category_transition_matrix")


---
## 8. Temporal Patterns

Temporal analysis reveals whether user and item activity is stationary or drifting — a prerequisite for deciding whether time-aware features and models are necessary.

| Plot | Analytical question |
|------|---------------------|
| Daily event volume | Is platform activity growing, declining, or seasonal? |
| Weekly active users & items | Are user and item counts co-evolving? |
| Category share drift | Are category preferences shifting over time? |
| Hour × Day-of-week heatmap | What are the intra-week and intra-day rhythms? |


In [ ]:
# ── 8a. Daily event volume ────────────────────────────────────────────────────
daily_vol = con.execute("""
    SELECT DATE(stime) AS day, COUNT(*) AS num_events
    FROM merrec GROUP BY day ORDER BY day
""").df()
daily_vol["day"] = pd.to_datetime(daily_vol["day"])

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(daily_vol["day"], daily_vol["num_events"], linewidth=0.9)
ax.set(yscale="log", xlabel="Date", ylabel="Events (log)",
       title="Daily Event Volume Over Time")
save("07a_daily_event_volume")


In [ ]:
# ── 8b. Weekly active users & items ──────────────────────────────────────────
weekly = con.execute("""
    SELECT
        DATE_TRUNC('week', stime)           AS week,
        APPROX_COUNT_DISTINCT(user_id)      AS users,
        APPROX_COUNT_DISTINCT(item_id)      AS items
    FROM merrec GROUP BY week ORDER BY week
""").df()
weekly["week"] = pd.to_datetime(weekly["week"])

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(weekly["week"], weekly["users"], label="Active users", linewidth=1.2)
ax.plot(weekly["week"], weekly["items"], label="Active items",  linewidth=1.2)
ax.set(yscale="log", xlabel="Week", ylabel="Count (log)",
       title="Weekly Active Users & Items")
ax.legend()
save("07b_weekly_active_users_items")


In [ ]:
# ── 8c. Category share drift (monthly, top-6 L1) ─────────────────────────────
top6_l1 = con.execute("""
    SELECT c1_name FROM (
        SELECT c1_name, COUNT(*) AS cnt
        FROM merrec WHERE c1_name IS NOT NULL
        GROUP BY c1_name ORDER BY cnt DESC LIMIT 6
    )
""").df()["c1_name"].tolist()

esc6   = [c.replace("'", "''") for c in top6_l1]
cats6  = "(" + ", ".join(f"'{c}'" for c in esc6) + ")"

cat_monthly = con.execute(f"""
    SELECT DATE_TRUNC('month', stime) AS month, c1_name, COUNT(*) AS cnt
    FROM merrec WHERE c1_name IN {cats6}
    GROUP BY month, c1_name ORDER BY month
""").df()
cat_monthly["month"] = pd.to_datetime(cat_monthly["month"])

pivot = cat_monthly.pivot_table(
    index="month", columns="c1_name", values="cnt", fill_value=0
).div(lambda df: df.sum(axis=1), axis=0)

# Manual normalisation (avoids lambda in older pandas)
pivot = pivot.div(pivot.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(11, 5))
ax.stackplot(pivot.index, pivot.T.values, labels=pivot.columns)
ax.set(xlabel="Month", ylabel="Category share",
       title="Category Share Drift Over Time (Top-6 L1 Categories)")
ax.legend(loc="upper right", fontsize=8)
save("07c_category_share_drift")


In [ ]:
# ── 8d. Hour-of-day × Day-of-week activity heatmap ───────────────────────────
heatmap_df = con.execute("""
    SELECT
        DAYOFWEEK(stime) AS dow,
        HOUR(stime)      AS hour,
        COUNT(*)         AS cnt
    FROM merrec
    GROUP BY dow, hour
""").df()

pivot_hm = heatmap_df.pivot_table(
    index="dow", columns="hour", values="cnt", fill_value=0
)
dow_labels = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot_hm, cmap="YlOrRd", ax=ax,
            linewidths=0.3, linecolor="white",
            yticklabels=dow_labels,
            cbar_kws={"label": "Event count"})
ax.set(xlabel="Hour of day", ylabel="Day of week",
       title="Activity Heatmap — Hour × Day of Week")
save("07d_activity_heatmap_hour_dow")


---
## 9. Price Distribution

Price is a key item-level feature. Its distribution informs feature scaling decisions and reveals segment structure across product categories. Price data is only available for a subset of interactions (see missing-value audit in Section 2).


In [ ]:
# ── 9a. Global price distribution ────────────────────────────────────────────
price_df = con.execute("""
    SELECT price FROM merrec WHERE price IS NOT NULL AND price > 0
""").df()

prices = price_df["price"].values

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(prices.clip(max=np.percentile(prices, 99)),
             bins=80, color=PALETTE[3], edgecolor="none", alpha=0.85)
axes[0].set(xlabel="Price (clipped at 99th percentile)",
            ylabel="Count",
            title="Price Distribution — Linear")

axes[1].hist(prices,
             bins=np.logspace(np.log10(prices.min() + 1e-3), np.log10(prices.max()), 60),
             color=PALETTE[3], edgecolor="none", alpha=0.85)
axes[1].set(xscale="log", yscale="log",
            xlabel="Price (log)",
            ylabel="Count (log)",
            title="Price Distribution — Log-Log")

fig.suptitle("Price Distribution", fontsize=13, fontweight="bold", y=1.02)
save("08_price_distribution")


In [ ]:
# ── 9b. Price by top-5 L0 category ───────────────────────────────────────────
top5_l0 = con.execute("""
    SELECT c0_name FROM (
        SELECT c0_name, COUNT(*) AS cnt
        FROM merrec WHERE c0_name IS NOT NULL
        GROUP BY c0_name ORDER BY cnt DESC LIMIT 5
    )
""").df()["c0_name"].tolist()

esc5   = [c.replace("'", "''") for c in top5_l0]
cats5  = "(" + ", ".join(f"'{c}'" for c in esc5) + ")"

price_cat = con.execute(f"""
    SELECT c0_name, price
    FROM merrec
    WHERE c0_name IN {cats5} AND price IS NOT NULL AND price > 0
""").df()

fig, ax = plt.subplots(figsize=(10, 5))
price_cat.boxplot(column="price", by="c0_name", ax=ax, sym="",
                  medianprops=dict(color="crimson", linewidth=2))
ax.set(yscale="log", xlabel="Category (L0)",
       ylabel="Price (log)", title="Price by Top-5 L0 Categories")
plt.suptitle("")
plt.xticks(rotation=20, ha="right")
save("08b_price_by_category")


---
## 10. Sparsity, Gini & Cold-Start Diagnostics

### Sparsity
The interaction matrix has dimensions $|\text{users}| \times |\text{items}|$. Sparsity is defined as:

$$\text{sparsity} = 1 - \frac{|\text{observed interactions}|}{|\text{users}| \times |\text{items}|}$$

In C2C datasets, sparsity typically exceeds 99.9%. This directly determines the feasibility of neighbourhood-based vs factorisation-based models.

### Gini Coefficient
The Gini coefficient measures interaction inequality on a scale of 0 (perfect equality) to 1 (one entity receives all interactions). A high Gini on the item side confirms the long-tail structure and informs how many items can realistically receive quality representations.

### Cold-Start
Cold-start users and items (those with ≤ $k$ interactions) are likely to receive poor latent representations. Quantifying their prevalence at multiple thresholds supports the research question on category-aware cold-start mitigation.


In [ ]:
# ── 10a. Gini coefficients & Lorenz curves ────────────────────────────────────
def gini(arr: np.ndarray) -> float:
    """Gini coefficient of a 1-D non-negative array."""
    arr = np.sort(arr.astype(float))
    n   = len(arr)
    idx = np.arange(1, n + 1)
    return float((2 * idx - n - 1).dot(arr) / (n * arr.sum()))

def lorenz(arr):
    arr = np.sort(arr.astype(float))
    cum = np.cumsum(arr)
    cum = np.insert(cum, 0, 0) / cum[-1]
    pop = np.linspace(0, 1, len(cum))
    return pop, cum

user_gini = gini(ec)
item_gini = gini(ic)
n_users   = len(ec)
n_items   = len(ic)
n_events  = int(ec.sum())
sparsity  = 1.0 - (n_events / (n_users * n_items))

print(f"User-side Gini  : {user_gini:.4f}")
print(f"Item-side Gini  : {item_gini:.4f}")
print(f"Matrix sparsity : {sparsity:.8f}")
print(f"Users           : {n_users:,}")
print(f"Items           : {n_items:,}")
print(f"Events          : {n_events:,}")


In [ ]:
pop_u, cum_u = lorenz(ec)
pop_i, cum_i = lorenz(ic)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(pop_u, cum_u, label=f"Users  (Gini = {user_gini:.3f})", linewidth=1.5)
ax.plot(pop_i, cum_i, label=f"Items  (Gini = {item_gini:.3f})", linewidth=1.5)
ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=0.8, label="Perfect equality")
ax.fill_between(pop_u, cum_u, pop_u, alpha=0.08)
ax.fill_between(pop_i, cum_i, pop_i, alpha=0.08)
ax.set(xlabel="Cumulative share of population",
       ylabel="Cumulative share of interactions",
       title="Lorenz Curves — Interaction Inequality")
ax.legend()
save("09_lorenz_gini")


In [ ]:
# ── 10b. Long-tail threshold analysis ────────────────────────────────────────
print(f"{'Label':<10} {'Coverage':>10} {'Threshold':>12} {'Entities':>12} {'% of events':>14}")
print("-" * 60)
for label, counts_arr in [("Users", ec), ("Items", ic)]:
    total_int = counts_arr.sum()
    for pct in [0.80, 0.90, 0.95]:
        threshold = np.percentile(counts_arr, (1 - pct) * 100)
        heavy_n   = int((counts_arr >= threshold).sum())
        heavy_int = int(counts_arr[counts_arr >= threshold].sum())
        print(f"{label:<10} {pct*100:>9.0f}%  {threshold:>12.0f}  {heavy_n:>12,}  {heavy_int/total_int*100:>13.1f}%")


In [ ]:
# ── 10c. Cold-start prevalence at multiple thresholds ─────────────────────────
thresholds = [1, 2, 5, 10, 20]
cold_user_rows = [(k, int((ec <= k).sum()), float((ec <= k).mean() * 100)) for k in thresholds]
cold_item_rows = [(k, int((ic <= k).sum()), float((ic <= k).mean() * 100)) for k in thresholds]

cold_user_df = pd.DataFrame(cold_user_rows, columns=["k", "n_users",  "pct"])
cold_item_df = pd.DataFrame(cold_item_rows, columns=["k", "n_items", "pct"])

print("Cold-start users (≤ k interactions):")
display(cold_user_df)
print("\nCold-start items (≤ k interactions):")
display(cold_item_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar([str(k) for k in thresholds], cold_user_df["pct"],
            color=PALETTE[0], edgecolor="none", alpha=0.85)
axes[0].set(xlabel="Interaction threshold k", ylabel="% of users",
            title="Cold-Start Users (≤ k interactions)")

axes[1].bar([str(k) for k in thresholds], cold_item_df["pct"],
            color=PALETTE[4], edgecolor="none", alpha=0.85)
axes[1].set(xlabel="Interaction threshold k", ylabel="% of items",
            title="Cold-Start Items (≤ k interactions)")

fig.suptitle("Cold-Start Prevalence by Threshold", fontsize=13, fontweight="bold", y=1.02)
save("10_cold_start_diagnostics")


In [ ]:
# ── 10d. Cold-start concentration by L0 category ──────────────────────────────
#
# Do cold-start items cluster in particular categories?
# If yes, category context becomes a stronger cold-start signal for those segments.

item_cat_counts = con.execute("""
    SELECT item_id, c0_name, COUNT(*) AS event_count
    FROM merrec WHERE c0_name IS NOT NULL
    GROUP BY item_id, c0_name
""").df()

cold_items = set(item_cat_counts.loc[item_cat_counts["event_count"] <= 5, "item_id"])

category_cold_share = (
    item_cat_counts
    .assign(is_cold=lambda df: df["item_id"].isin(cold_items))
    .groupby("c0_name")["is_cold"]
    .mean()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)
category_cold_share.columns = ["c0_name", "cold_item_share"]

fig, ax = plt.subplots(figsize=(9, max(5, len(category_cold_share) * 0.38)))
ax.barh(category_cold_share["c0_name"][::-1],
        category_cold_share["cold_item_share"][::-1] * 100,
        color=PALETTE[5], edgecolor="none", alpha=0.85)
ax.set(xlabel="Cold-start item share (%, ≤ 5 events)",
       title="Cold-Start Item Concentration by Category (Top-20)")
save("10b_cold_start_by_category")


---
## Summary & Next Steps

This notebook has characterised the MerRec dataset across all structural, behavioral, temporal, and distributional dimensions required for Layer 1.

### Key findings to carry forward

| Dimension | Finding | Layer 2 implication |
|-----------|---------|---------------------|
| User activity | Strong long-tail (high Gini) | RFM features; activity-tier flags |
| Item activity | Stronger long-tail than users | Item popularity buckets; cold-start flags |
| Session structure | High bounce rate; bi-modal session lengths | Bounce indicator; session depth features |
| Category transitions | Strong self-loop tendency | Category stickiness feature |
| Temporal | Non-stationary volume; intra-week patterns | Time-decay weights; hour/DOW features |
| Cold-start | Significant fraction at ≤5 events | Category-context embeddings in Layer 3 |

### Next: Layer 2 — Feature Engineering

The findings above directly inform the feature engineering pipeline:
- **User-level**: recency, frequency, monetary (RFM), category entropy, session count, active lifespan
- **Item-level**: popularity bucket, freshness, price tier, cold-start flag, category level
- **Session-level**: length, duration, item breadth, bounce flag, event diversity


In [ ]:
# Close DuckDB connection cleanly
con.close()
print("DuckDB connection closed.")
print(f"All figures saved to: {OUTPUT_DIR.resolve()}")
